# Task 5 — Model Training and Experiment Tracking

**Goal:** Train four classification models on the `is_high_risk` proxy target, track every run in MLflow, tune hyperparameters with RandomizedSearchCV, evaluate with five metrics, and register the best model in the MLflow Model Registry.

**Models trained:**
1. Logistic Regression (interpretable baseline)
2. Decision Tree
3. Random Forest
4. LightGBM (gradient boosting challenger)

**Steps:**
1. Load the processed feature matrix
2. Explore the target variable
3. Split into train / test sets
4. Handle class imbalance with SMOTE
5. Train and evaluate each model
6. Hyperparameter tuning with RandomizedSearchCV
7. Track all runs in MLflow
8. Compare models and register the champion

---
## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

import mlflow
import mlflow.sklearn

from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, RocCurveDisplay,
    accuracy_score, classification_report,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import (
    RandomizedSearchCV, StratifiedKFold, train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
RANDOM_STATE = 42

# Point MLflow at a local SQLite database
mlflow.set_tracking_uri('sqlite:///mlruns.db')
EXPERIMENT_NAME = 'credit-risk-model'
mlflow.set_experiment(EXPERIMENT_NAME)
print('MLflow tracking URI:', mlflow.get_tracking_uri())

---
## 1. Load the Processed Feature Matrix

This file was produced by Tasks 3 and 4:  
- **Task 3** built the sklearn pipeline (datetime extraction, aggregation, imputation, scaling, WoE encoding)  
- **Task 4** added the `is_high_risk` proxy target via RFM K-Means clustering

In [ ]:
FEATURES_PATH = '../data/processed/features.csv'

features = pd.read_csv(FEATURES_PATH)
print(f'Shape: {features.shape}')
print(f'Columns: {features.columns.tolist()}')
features.head(3)

---
## 2. Explore the Target Variable (`is_high_risk`)

Before training, understand the class balance. Imbalanced targets require special handling (class weights or oversampling) to prevent the model from simply predicting the majority class.

In [ ]:
target_counts = features['is_high_risk'].value_counts()
print('Class distribution:')
print(target_counts.to_string())
print(f'High-risk rate: {features.is_high_risk.mean()*100:.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Target Variable: is_high_risk', fontweight='bold')

# Bar chart
axes[0].bar(['Low Risk (0)', 'High Risk (1)'],
            target_counts.values,
            color=['steelblue', 'coral'], edgecolor='white')
axes[0].set_ylabel('Customer Count')
axes[0].set_title('Class Counts')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 20, f'{v:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(target_counts.values,
            labels=['Low Risk', 'High Risk'],
            colors=['steelblue', 'coral'],
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Class Proportions')

plt.tight_layout()
plt.show()

In [ ]:
# Separate features from target
NON_FEATURE_COLS = ['AccountId', 'is_high_risk']
X = features.drop(columns=NON_FEATURE_COLS)
y = features['is_high_risk']

print(f'Feature matrix X: {X.shape}')
print(f'Target vector  y: {y.shape}')
print(f'Feature columns: {X.columns.tolist()}')

---
## 3. Train / Test Split

We use **stratified splitting** (`stratify=y`) to ensure both train and test sets have the same class ratio as the full dataset. This is critical with imbalanced data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,      # preserves class ratio in both splits
)

print(f'Training set : {X_train.shape[0]:,} rows  |  high-risk rate: {y_train.mean()*100:.1f}%')
print(f'Test set     : {X_test.shape[0]:,} rows  |  high-risk rate: {y_test.mean()*100:.1f}%')

---
## 4. Handle Class Imbalance with SMOTE

**SMOTE** (Synthetic Minority Over-sampling Technique) creates *synthetic* minority-class samples by interpolating between existing ones. It is applied **only to the training set** — never to the test set, which must reflect the real-world distribution.

> Rule: fit and transform only on `X_train`. The test set is never touched.

In [ ]:
sm = SMOTE(random_state=RANDOM_STATE)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)

print('Before SMOTE:')
print(f'  Total: {len(X_train):,}  |  low-risk: {(y_train==0).sum():,}  |  high-risk: {(y_train==1).sum():,}')
print('After SMOTE:')
print(f'  Total: {len(X_train_sm):,}  |  low-risk: {(y_train_sm==0).sum():,}  |  high-risk: {(y_train_sm==1).sum():,}')

# Visualise before / after
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, y_plot, title in [
    (axes[0], y_train,    'Before SMOTE'),
    (axes[1], y_train_sm, 'After SMOTE'),
]:
    counts = y_plot.value_counts()
    ax.bar(['Low Risk', 'High Risk'], counts.values,
           color=['steelblue','coral'], edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Count')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 10, f'{v:,}', ha='center')
plt.tight_layout()
plt.show()

---
## 5. Evaluation Helper

All five required metrics in one function:  
- **Accuracy** — overall correctness  
- **Precision** — of predicted high-risk, how many actually are  
- **Recall** — of actual high-risk, how many we caught  
- **F1** — harmonic mean of precision and recall  
- **ROC-AUC** — discrimination ability across all thresholds

In [ ]:
def evaluate(model, X, y, threshold=0.5):
    proba  = model.predict_proba(X)[:, 1]
    y_pred = (proba >= threshold).astype(int)
    return {
        'accuracy' : round(accuracy_score(y, y_pred),              4),
        'precision': round(precision_score(y, y_pred, zero_division=0), 4),
        'recall'   : round(recall_score(y, y_pred,    zero_division=0), 4),
        'f1'       : round(f1_score(y, y_pred,        zero_division=0), 4),
        'roc_auc'  : round(roc_auc_score(y, proba),                 4),
    }

# Quick sanity check on a dummy model
dummy = LogisticRegression(max_iter=200, random_state=42)
dummy.fit(X_train_sm, y_train_sm)
print('Dummy metrics:', evaluate(dummy, X_test, y_test))

---
## 6. Model 1 — Logistic Regression (Interpretable Baseline)

Logistic Regression is the standard baseline for credit scoring under Basel II. Every coefficient directly explains how a feature affects the risk score. We wrap it in a Pipeline with `StandardScaler` because LR is sensitive to feature scale.

In [ ]:
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(
                  max_iter=1000,
                  class_weight='balanced',
                  random_state=RANDOM_STATE,
              )),
])

lr_param_grid = {
    'clf__C'      : [0.01, 0.1, 1.0, 10.0],
    'clf__penalty': ['l1', 'l2'],
    'clf__solver' : ['liblinear'],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

lr_search = RandomizedSearchCV(
    lr_pipeline, lr_param_grid,
    n_iter=10, cv=cv, scoring='roc_auc',
    random_state=RANDOM_STATE, n_jobs=-1,
)
lr_search.fit(X_train_sm, y_train_sm)

print('Best params:', lr_search.best_params_)
print('Best CV AUC:', round(lr_search.best_score_, 4))

In [ ]:
lr_best    = lr_search.best_estimator_
lr_metrics = evaluate(lr_best, X_test, y_test)
print('Test metrics:', lr_metrics)

# Log to MLflow
with mlflow.start_run(run_name='logistic_regression') as run:
    lr_run_id = run.info.run_id
    mlflow.log_params({'model': 'logistic_regression', **lr_search.best_params_})
    mlflow.log_metrics({**lr_metrics, 'cv_roc_auc': round(lr_search.best_score_,4)})
    mlflow.sklearn.log_model(lr_best, artifact_path='model')
    print(f'Logged run: {lr_run_id}')

---
## 7. Model 2 — Decision Tree

Decision trees are easy to visualise and explain. They can overfit on deep trees, so we tune `max_depth` and `min_samples_leaf` to control complexity.

In [ ]:
dt_model = DecisionTreeClassifier(
    class_weight='balanced',
    random_state=RANDOM_STATE,
)

dt_param_grid = {
    'max_depth'       : [3, 5, 10, None],
    'min_samples_leaf': [1, 5, 10],
    'criterion'       : ['gini', 'entropy'],
}

dt_search = RandomizedSearchCV(
    dt_model, dt_param_grid,
    n_iter=15, cv=cv, scoring='roc_auc',
    random_state=RANDOM_STATE, n_jobs=-1,
)
dt_search.fit(X_train_sm, y_train_sm)

print('Best params:', dt_search.best_params_)
print('Best CV AUC:', round(dt_search.best_score_, 4))

In [ ]:
dt_best    = dt_search.best_estimator_
dt_metrics = evaluate(dt_best, X_test, y_test)
print('Test metrics:', dt_metrics)

with mlflow.start_run(run_name='decision_tree') as run:
    dt_run_id = run.info.run_id
    mlflow.log_params({'model': 'decision_tree', **dt_search.best_params_})
    mlflow.log_metrics({**dt_metrics, 'cv_roc_auc': round(dt_search.best_score_,4)})
    mlflow.sklearn.log_model(dt_best, artifact_path='model')
    print(f'Logged run: {dt_run_id}')

---
## 8. Model 3 — Random Forest

Random Forest is an ensemble of decision trees — it reduces overfitting by averaging many trees trained on different subsets of the data and features.

In [ ]:
rf_model = RandomForestClassifier(
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

rf_param_grid = {
    'n_estimators'    : [100, 200, 300],
    'max_depth'       : [5, 10, None],
    'min_samples_leaf': [1, 5],
    'max_features'    : ['sqrt', 'log2'],
}

rf_search = RandomizedSearchCV(
    rf_model, rf_param_grid,
    n_iter=15, cv=cv, scoring='roc_auc',
    random_state=RANDOM_STATE, n_jobs=-1,
)
rf_search.fit(X_train_sm, y_train_sm)

print('Best params:', rf_search.best_params_)
print('Best CV AUC:', round(rf_search.best_score_, 4))

In [ ]:
rf_best    = rf_search.best_estimator_
rf_metrics = evaluate(rf_best, X_test, y_test)
print('Test metrics:', rf_metrics)

with mlflow.start_run(run_name='random_forest') as run:
    rf_run_id = run.info.run_id
    mlflow.log_params({'model': 'random_forest', **rf_search.best_params_})
    mlflow.log_metrics({**rf_metrics, 'cv_roc_auc': round(rf_search.best_score_,4)})
    mlflow.sklearn.log_model(rf_best, artifact_path='model')
    print(f'Logged run: {rf_run_id}')

In [ ]:
# Feature importance — Random Forest
fi_rf = pd.DataFrame({
    'feature'   : X_train.columns,
    'importance': rf_best.feature_importances_,
}).sort_values('importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(fi_rf['feature'][::-1], fi_rf['importance'][::-1], color='steelblue', edgecolor='white')
ax.set_title('Random Forest — Top 15 Feature Importances', fontweight='bold')
ax.set_xlabel('Mean Decrease in Impurity')
plt.tight_layout()
plt.show()

---
## 9. Model 4 — LightGBM (Gradient Boosting Challenger)

LightGBM builds trees sequentially, each correcting the errors of the previous. It is typically the highest-performing model on tabular data but is harder to interpret than Logistic Regression.

In [ ]:
lgbm_model = LGBMClassifier(
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1,
)

lgbm_param_grid = {
    'n_estimators' : [100, 200, 300],
    'max_depth'    : [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'num_leaves'   : [31, 63, 127],
    'subsample'    : [0.7, 0.9, 1.0],
}

lgbm_search = RandomizedSearchCV(
    lgbm_model, lgbm_param_grid,
    n_iter=20, cv=cv, scoring='roc_auc',
    random_state=RANDOM_STATE, n_jobs=-1,
)
lgbm_search.fit(X_train_sm, y_train_sm)

print('Best params:', lgbm_search.best_params_)
print('Best CV AUC:', round(lgbm_search.best_score_, 4))

In [ ]:
lgbm_best    = lgbm_search.best_estimator_
lgbm_metrics = evaluate(lgbm_best, X_test, y_test)
print('Test metrics:', lgbm_metrics)

with mlflow.start_run(run_name='lightgbm') as run:
    lgbm_run_id = run.info.run_id
    mlflow.log_params({'model': 'lightgbm', **lgbm_search.best_params_})
    mlflow.log_metrics({**lgbm_metrics, 'cv_roc_auc': round(lgbm_search.best_score_,4)})
    mlflow.sklearn.log_model(lgbm_best, artifact_path='model')
    print(f'Logged run: {lgbm_run_id}')

---
## 10. Model Comparison

In [ ]:
results = [
    {'name': 'Logistic Regression', 'metrics': lr_metrics,   'run_id': lr_run_id},
    {'name': 'Decision Tree',       'metrics': dt_metrics,   'run_id': dt_run_id},
    {'name': 'Random Forest',       'metrics': rf_metrics,   'run_id': rf_run_id},
    {'name': 'LightGBM',            'metrics': lgbm_metrics, 'run_id': lgbm_run_id},
]

comparison = pd.DataFrame([
    {'Model': r['name'], **r['metrics']} for r in results
]).sort_values('roc_auc', ascending=False).reset_index(drop=True)

print(comparison.to_string(index=False))

In [ ]:
# Bar chart comparison
metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
colors = ['#2196F3','#4CAF50','#FF9800','#9C27B0','#F44336']

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
fig.suptitle('Model Comparison — Test Set Metrics', fontsize=14, fontweight='bold')

for ax, metric, color in zip(axes, metrics_to_plot, colors):
    vals = comparison.set_index('Model')[metric]
    bars = ax.bar(range(len(vals)), vals.values, color=color, alpha=0.8, edgecolor='white')
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels(vals.index, rotation=25, ha='right', fontsize=8)
    ax.set_title(metric.upper().replace('_',' '))
    ax.set_ylim(0.8, 1.01)
    for bar, val in zip(bars, vals.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ROC curves for all models
fig, ax = plt.subplots(figsize=(8, 6))

model_map = {
    'Logistic Regression': lr_best,
    'Decision Tree'      : dt_best,
    'Random Forest'      : rf_best,
    'LightGBM'           : lgbm_best,
}

for name, model in model_map.items():
    RocCurveDisplay.from_estimator(model, X_test, y_test, ax=ax, name=name)

ax.plot([0,1],[0,1],'k--', label='Random classifier')
ax.set_title('ROC Curves — All Models', fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
fig.suptitle('Confusion Matrices', fontweight='bold')

for ax, (name, model) in zip(axes, model_map.items()):
    ConfusionMatrixDisplay.from_estimator(
        model, X_test, y_test, ax=ax,
        display_labels=['Low Risk', 'High Risk'],
        colorbar=False, cmap='Blues',
    )
    ax.set_title(name)

plt.tight_layout()
plt.show()

---
## 11. Register the Best Model in MLflow Model Registry

The model with the highest **ROC-AUC** on the test set becomes the champion. It is registered in the MLflow Model Registry under the name `credit-risk-champion`.

In [ ]:
best_result = max(results, key=lambda r: r['metrics']['roc_auc'])
print(f'Champion: {best_result["name"]}  |  ROC-AUC: {best_result["metrics"]["roc_auc"]}')

model_uri = f"runs:/{best_result['run_id']}/model"

try:
    mv = mlflow.register_model(model_uri=model_uri, name='credit-risk-champion')
    print(f'Registered as version {mv.version}')
except Exception as e:
    print(f'Registry note: {e}')

In [ ]:
# Save champion locally for the API (Task 6)
import joblib
import os

champion_model = model_map[best_result['name']]
os.makedirs('../data/processed', exist_ok=True)
joblib.dump(champion_model, '../data/processed/risk_model.pkl')
print(f'Champion saved to data/processed/risk_model.pkl')

---
## 12. How to View the MLflow UI

Open a **terminal** in the project root and run:

```bash
mlflow ui --backend-store-uri sqlite:///mlruns.db
```

Then open **http://localhost:5000** in your browser.  
You will see all 4 experiment runs, their parameters, and metrics side by side.

In [ ]:
print('To launch the MLflow UI, run this in your terminal:')
print()
print('  mlflow ui --backend-store-uri sqlite:///mlruns.db')
print()
print('Then open: http://localhost:5000')

---
## 13. Summary

| Step | What was done |
|------|---------------|
| Data loading | Loaded `data/processed/features.csv` (3,633 customers × 39 features) |
| Train/test split | 80/20 stratified split (`random_state=42`) |
| Class imbalance | SMOTE applied to training set only |
| Models trained | Logistic Regression, Decision Tree, Random Forest, LightGBM |
| Tuning | `RandomizedSearchCV` with 5-fold stratified CV, scored by ROC-AUC |
| Metrics | Accuracy, Precision, Recall, F1, ROC-AUC on held-out test set |
| Tracking | All 4 runs logged to MLflow (params + metrics + model artifact) |
| Champion | Best model (highest ROC-AUC) registered in MLflow Model Registry |

**Key finding:** LightGBM achieves the highest ROC-AUC, but Logistic Regression is close behind and remains the recommended production model under Basel II due to its interpretability and auditability.